In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [3]:
from IPython.display import clear_output
from src.dataset_loaders_new import get_samplers, get_indices
from src.utils import get_pca_models
from src import utils
from src.train import train_continuous
import wandb
from torch.utils.data import TensorDataset, DataLoader
import yaml
import numpy as np
import pickle
import random
import jax

## 1. Parameters.

Possible ```DATASET_NAME``` values are: ```twitter```, ```wiki-gigaword```, ```bone_marrow```

In [4]:

DATASET_NAME = 'twitter'
METHOD_NAME  = 'FlowGW'
DEVICE       = 'cuda:0'
SOURCE_DIM   = 50  #38 (bone_marrow)
TARGET_DIM   = 25 #50 (bone_marrow)
ALPHA        = 0.5
SEED         = 43
TEST_SPLIT   = 0.9

config = {'dataset':dict(DATASET_NAME     = DATASET_NAME,
                         DEVICE           = DEVICE,
                         SOURCE_DIM       = SOURCE_DIM,
                         TARGET_DIM       = TARGET_DIM,
                         N_MAX_SAMPLES    = 400000, #set to 6667 to get N_train=3K
                         N_TRAIN_SAMPLES  = 360000, #We used 6000 for the others
                         N_TEST_SAMPLES   = 512,
                         N_EVAL           = 2,
                         ALPHA            = ALPHA, 
                         BATCH_SIZE_TRAIN = 512,
                         BATCH_SIZE_TEST  = 512,
                         SEED             = SEED,
                         TRAIN_TYPE       = 'continuous',
                         NORMALIZE_VECS   = False
                          ),
          
          'training':dict(METHOD_NAME     = METHOD_NAME,
                          N_EPOCHS        = 12000,
                          ),
          
          #'model_specific':dict(HIDDEN_SIZES_MLP= [512, 256, 256],
          #                      EPS_FIT         = 0.01,
          #                      EPS_REG         = 0.001,
          #                      LAMBDA          = 1,
          #                      MOVER_LR        = 1e-4,
          'model_specific':dict(HIDDEN_SIZES_MLP = [1024, 1024, 1024, 1024],
                                EPS              = 1e-3,
                                N_FREQ           = 128,
                                MOVER_LR         = 1e-4
           )}


## 2. Loading dataset.

In [5]:
dataset_path = '../datasets'
sys.path.append(dataset_path)

source_vectors, target_vectors, random_indices_train, random_indices_test = get_indices(dataset_path, config)
source_vectors, target_vectors, train_sampler, test_sampler = get_samplers(source_vectors, target_vectors, random_indices_train, random_indices_test, config)

print(source_vectors.shape)
print(target_vectors.shape)

Loading model twitter_50 to source...
Loading model twitter_25 to target...
Source pairs...
180000
Target pairs...
180000
torch.Size([400000, 50])
torch.Size([400000, 25])


## 3. Training.

In [ ]:
import warnings


warnings.simplefilter(action='ignore', category=FutureWarning)
n_repeats = 10
wandb_report = True
project_name = f'{METHOD_NAME}_{DATASET_NAME}_{SOURCE_DIM}->{TARGET_DIM}_{n_repeats}reps_400K_fixed'

metric_names_train = ['Top@1', 'Top@5', 'Top@10', 'cossim_gt', 'inner_gw', 'foscttm']

#alpha_values = [0.0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0][::-1]
alpha_values = [0.2 ,0.9, 1.0][::-1]

metrics_out = {str(np.round(alpha, 1)):[] for alpha in alpha_values}

for ALPHA in alpha_values:
    
    config['dataset']['ALPHA'] = ALPHA 
    
    if ALPHA == 1.0:
        config['training']['N_EPOCHS'] = 12000
    else:
        config['training']['N_EPOCHS'] = 3000 
  
    if ALPHA == 1.0:
        n_repeats = 3
    else:
        n_repeats = 5 
        
    print('================================')
    print(f'Experiment for ALPHA={ALPHA}')
    print('================================')
    source_vectors, target_vectors, train_sampler, test_sampler = get_samplers(source_vectors, target_vectors, random_indices_train, random_indices_test, config)
    source_vectors, target_vectors, random_indices_train, random_indices_test = get_indices(dataset_path, config)
    
    for ix in range(n_repeats):

        SEED = random.randint(0, 10000)
        rng = jax.random.PRNGKey(SEED)#utils.default_prng_key([number, number])
        config['dataset']['SEED'] = SEED
        print('Seed: ', SEED)
        
        if wandb_report:
            exp_name = f'ALPHA_{np.round(ALPHA, 1)}_repeat_{ix}'
            wandb.init(name=exp_name, config=config, project=project_name)
        
        trained_class, metrics_dict = train_continuous(train_sampler, test_sampler, 
                                                       metric_names_train, target_vectors,
                                                       config,
                                                       wandb_report=wandb_report,
                                                       axis_lims=None, report_every=20000)
        
        metrics_out[str(np.round(ALPHA, 1))].append(metrics_dict)
        #print(metrics_out)
    
        with open(f'results_continuous/{project_name}.pkl', 'wb') as f:
            pickle.dump(metrics_out, f)

Experiment for ALPHA=1.0
Source pairs...
180000
Target pairs...
180000
Loading model twitter_50 to source...
Loading model twitter_25 to target...


2024-10-02 07:15:26.742446: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.3 which is older than the ptxas CUDA version (12.5.40). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


Seed:  4540


wandb: Currently logged in as: xavier13091994 (entropic_gw). Use `wandb login --relogin` to force relogin


Epoch:   0%|          | 0/12000 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/2 [00:00<?, ?it/s]

Seed:  2998


test/Top@1,▁
test/Top@10,▁
test/Top@5,▁
test/cossim_gt,▁
test/foscttm,▁
test/inner_gw,▁
test/step,▁▁▁▁▁▁
train/Top@1,▁
train/Top@10,▁
train/Top@5,▁
train/cossim_gt,▁


Epoch:   0%|          | 0/12000 [00:00<?, ?it/s]